# LETKF Cycling Capstone Project
***


## Introduction

Throughout this tutorial series you've built up experience running the LETKF on a quasi-geostrophic (QG) model in JEDI — you've seen how ensemble size, localization, and inflation each shape the analysis in different ways. Now it's time to put all of that together.

Your goal here is simple: **tune the LETKF to get the best cycling performance you can.** There's no single right answer, and there are many paths to a good result. You are being put into a position of a researcher trying to tune their DA system, subject to computational or time restraints.

You will run **200 cycles** of the LETKF with a **reduced resolution** ensemble to mimic an 'imperfect model' setup - the model is run at 20 x 10 x 2 resolution, coarser than the 40 x 20 x 2 truth simulation. It is a more difficult problem than the cycling we have performed already, since our model will contain substantial error compared to the truth. Can you find a stable LETKF cycling scenario in this case?

All observation and ensemble background files have already been provided at `/home/nonroot/shared/EDU/CAPSTONE/data`; these will be the data used for your experiments here.

### What You Can Tune

You have four knobs to play with, all set in the **DA EXPERIMENT & PARAMETERS DEFINITION SECTION** cell just below this section:

| Parameter | Variable | Default | What it does |
|---|---|---|---|
| Ensemble size | `ENSSIZE` | 20 | More members = better sampling of uncertainty, but each one costs you compute time |
| Localization scale | `LOCSCALE` | 4.0 × 10⁶ m | How far away observations can reach — too wide and you get sampling noise, too narrow and you throw away useful obs |
| RTPP inflation | `RTPPINFL` | 0.5 (range 0–1) | Blends analysis perturbations back toward the prior to recover spread lost in the update step |
| Multiplicative inflation | `MULTINFL` | 1.1 (generally > 1) | Uniformly scales ensemble perturbations after the update to counter systematic underdispersion |

Feel free to explore these in whatever order or combination makes sense to you. Try big changes, try small ones — build your own intuition for how these parameters interact.

> #### Note on Parameter Choices
> - Keep ENSSIZE ≤ 45. There is a known issue with JEDI and higher ensemble member counts that can cause unrealistic results (i.e. (error grows with each cycle)
> - You may find some combinations produce unrealistic results (e.g. error grows with each cycle), or even encounter a crash in the LETKF or forecast steps. For example, a high LOCSCALE coupled with a high ENSSIZE has been known to cause this issue with our tests. If this happens, you wil need to modify one of the parameters an run a new experiment

### How You'll Be Evaluated

Your final experiment must run for the **full 200 cycles**. 

You will submit your produced average RMSE and ensemble variance text files (`rmse_stream.txt` and `ensvar_stream.txt`) as well as parameter choices.

Two diagnostics will be computed by the organizers from these streamfunction error and variance files, verified against the truth:

1. **Average Prior RMSE in Streamfunction** — lower is better. How well does your background track the true state over 200 cycles?

2. **Spread–Skill Ratio (SSR)** — This is the ratio of the average prior spread to the prior RMSE. We want to aim for ≈ 1.0. Below 1 means your ensemble is overconfident and underdispersed; above 1 means it's underconfident and overdispersed.

The challenge is that these two goals can pull in different directions — getting low RMSE *and* well-calibrated spread at the same time takes some care.

### Tips for Managing Your Time and Number of Cycles

 - Use at least 10 cycles for tuning experiments, to allow enough time to "spin up" the ensemble spread to a stable place.
 - For tuning experiments, more cycles used gives a better sample; however, the computation cost for each tuning experiment will be higher. With  10 cycles, the tuning experiments will take anywhere from **~2-4 minutes** depending on parameter choices. This is **doubled** for 20 cycles, though it still may be worth the added time as you see the trends of the sawtooths change with higher cycle counts. **The choice of how many cycles to use for tuning is ultimately up to you.** 
 - You will be evaulated on a full run using <u>200 cycles</u>. **Be mindful of how long this will take before submission.** Depending on your final parameter choices, a full 200-cycle run could take anywhere from **1-2 hours** or possibly longer to complete in total.
 - You can run more than one experiment with 200 cycles if you desire, but you may only submit **one** experiment's results  

### Running and Evaluating an Experiment

1. The workflow is straightforward: just change the values in the **DA Experiment & Parameters Definition** cell just below, give your experiment a descriptive name via `EXPNAME`, and run all the cells. You'll get RMSE sawtooth plots with ensemble spread overlaid. Good luck, and have fun with it!
2. Results from your experiments will be placed in automatically created folders, located at `/home/nonroot/shared/EDU/CAPSTONE/output/$EXPNAME`, where $EXPNAME is the name of the experiment you set in the cell below along with other parameters. Navigate to that directory, and then into `plots` to see the results of your experiment.
3. Try the `exp_default_params` experiment first with 10 cycles, verify that you can produce the same sawtooth and spread diagram as is pasted here before you start with your own tuning experiments! The rmse plot will be found at `/home/nonroot/shared/EDU/CAPSTONE/output/exp_default_params/plots`, which you can navigate to using the directory tree to the left.

<center><img src="images/rmse_sawtooth_default.png" alt="sawtooth_default" width="800"/></center>


### Set Environment Variables 


In [ ]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU

### DA EXPERIMENT & PARAMETERS DEFINITION SECTION
**Set the DA parameters below for individual experiments.** These will be dynamically inserted into the yaml files on-the-fly for all subsequent LETKF and forecast yamls. <u>Warning: Do not change the names of the exported variables (EXPNAME, ENSSIZE, etc.). Only change the values they are being set to.</u>
> #### Note on experiment name (EXPNAME)
>Give your experiment a unique identifying name for the output results directory. One suggestion is to incorporate parameter choice into the name in some way (e.g. "exp_loc4" if you set localization to 4.0e6).
But ultimately, it is your choice!

In [ ]:
# Recommend a smaller number (~10-40) for tuning experiments.
# ** BUT MUST SET TO 200 FOR FINAL TUNED EXPERIMENT **
export ncycles=10

# Experiment name  (default is "exp_default_params")
export EXPNAME="exp_default_params"

# Ensemble Size (default is 20; do not set higher than 45)
export ENSSIZE=20

# Localization scale in meters (default is "4.0e6" in meters, 
# you can set to any value but keep in mind grid spacing 
# is approximately 1000 x 500 km for the truth and 2000 x 1000 km for reduced resolution background ensemble)
export LOCSCALE="4.0e6"

# Relaxation to Prior Perturbation (RTPP) inflation blending amount (default is 0.5, range of values is 0.0-1.0)
export RTPPINFL="0.5"

# Multiplicative inflation amount (default is 1.1, generally set to some value > 1)
export MULTINFL="1.1"


In [ ]:
##### First, need to create experiment output and yaml directories
cd $JEDI_EDU
export EXPDIR="./CAPSTONE/output/${EXPNAME}"
if [ -d "$EXPDIR" ]; then
   echo "Removing old directory first! ${EXPDIR}"
   rm -rf $EXPDIR
fi
mkdir -vp ${EXPDIR}/da
mkdir -vp ${EXPDIR}/logs 
mkdir -vp ${EXPDIR}/plots

# Experiment Running Section
***

### Perform LETKF and QG forecast Cycling
- This setup uses one big loop for all cycles. Yamls are made on-the-fly for LETKF and Qg forecast components, using template yaml files `letkf_template_capstone.yaml` and `forecast_template_capstone.yaml`, in the `CAPSTONE/yamls` directory. <br>
- Error checks included throughout, and loop will stop if an error encountered. If this happens, examine the output log, or change the DA parameters and run a new experiment
- Comments included below for different parts of this cycling loop. Try to get a basic understanding of how everything works together!

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate    # Initialize date looping variable for each cycle
start_time=`date +%s` # Needed to calculate time it takes to complete whole cycling
no_err=true
for icyc in $(seq 1 $ncycles); do
   start_time_cyc=`date +%s`

   # Set some date variables needed for yaml files, based on current loop's date (curdate)
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")
   if [ $icyc -eq 1 ]; then
      # Initial conditions file for first cycle (pre-generated background ensemble)
      bgfile="CAPSTONE/data/bg/bkgd_reduced.ens.%mem%.${curdate_minus1day}.P1D.nc"
   else
      # Initial conditions file for subsequent cycles (1-day forecast using previous letkf analysis)
      bgfile="${EXPDIR}/da/letkf.%mem%.fc.${curdate_minus1day}.P1D.nc"
   fi

   # Create LETKF yaml for this cycle. 
   # This is where the values set at top of notebook are inserted into the yamls dynamically
   sed -e "s|@DATE_BGN|${curdate_minus6h}|g;s|@DATE_ANL|${curdate}|g" \
       -e "s|@CYC|$icyc|g;s|@BACKGROUND_FILE|${bgfile}|g" \
       -e "s|@EXPNAME|$EXPNAME|g;s|@ENSSIZE|${ENSSIZE}|g" \
       -e "s|@LOCSCALE|$LOCSCALE|g;s|@RTPPINFL|${RTPPINFL}|g;s|@MULTINFL|${MULTINFL}|g" \
       ./CAPSTONE/yamls/letkf_template_capstone.yaml > ./CAPSTONE/yamls/letkf_cyc${icyc}.yaml

   # Run LETKF!
   $JEDI_BIN/qg_letkf.x ./CAPSTONE/yamls/letkf_cyc${icyc}.yaml >& ${EXPDIR}/logs/letkf_cyc${icyc}.log
   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running LETKF for cycle $icyc."
      echo "Check ${EXPDIR}/logs/letkf_cyc${icyc}.log"
      echo "Stopping loop"
      no_err=false
      break
   else
      end_time_letkf=`date +%s`
      echo "SUCCESS LETKF run for cycle $icyc, took $(( end_time_letkf - start_time_cyc )) seconds"
   fi


   # Then run QG ensemble forecast to next cycle by looping over each member
   for imem in $(seq 1 ${ENSSIZE}); do
   
      # First create forecast yaml for ensemble forecast to next cycle
      sed -e "s|%MEMBER%|$(printf "%06d\n" "$imem")|g;s|%MEM%|$imem|g" \
          -e "s|@EXPNAME|${EXPNAME}|g;s|@DATE_IC|${curdate}|g" \
             ./CAPSTONE/yamls/forecast_template_capstone.yaml \
           > ./CAPSTONE/yamls/forecast_cyc${icyc}_mem${imem}.yaml
   
      # Run QG model forecast for member
      $JEDI_BIN/qg_forecast.x ./CAPSTONE/yamls/forecast_cyc${icyc}_mem${imem}.yaml \
                           >& ${EXPDIR}/logs/forecast_cyc${icyc}_mem${imem}.log
      if [ $? -ne 0 ]; then
         echo "ERROR! Something went wrong running forecast for cycle $icyc, member $imem."
         echo "Check ${EXPDIR}/logs/forecast_cyc${icyc}_mem${imem}.log"
         echo "Stopping loop"
         no_err=false
         break 2  # The "2" specifies to break out of two loops (member and cycle loops)
      else
         printfreq=10 # How often (after how many member forecasts) to print success messages, to see progress
         if [ $((imem%printfreq)) -eq 0 ]; then 
            echo "SUCCESS forecast run for cycle $icyc, members $((imem-printfreq+1))-$imem"
         fi
         # Remove yaml (keep first member as an example)
         [[ ${imem} -ne 1 ]] && rm ./CAPSTONE/yamls/forecast_cyc${icyc}_mem${imem}.yaml
      fi
      
   done # Member loop (imem)
   end_time_fcst=`date +%s`
   echo "SUCCESS ensemble forecast run for cycle $icyc, took $(( end_time_fcst - end_time_letkf )) seconds" 

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
   
done # Cycle loop (icyc)

end_time_total=`date +%s`
if [ "$no_err" = true ]; then
   echo "SUCCESS completion of $ncycles cycles for experiment $EXPNAME took $(( end_time_total - start_time )) seconds"
fi

#### Verify results were produced
- Confirm files were produced for all cycles

In [ ]:
cd $JEDI_EDU
echo "Ensemble Member 1 Analyses (used as ICs for member 1 forecast to the next cycle):"
ls ${EXPDIR}/da/letkf.000001.an*nc
echo -e "\nEnsemble Member 1 Forecasts (used as backgrounds for each cycle):"
ls ${EXPDIR}/da/letkf.1.fc*P1D.nc
echo -e "\nEnsemble Mean Analyses:"
ls ${EXPDIR}/da/letkf.000000.an*nc
echo -e "\nEnsemble Mean Prior and Increments:"
ls ${EXPDIR}/da/letkf_mean*nc


### Compute Ensemble Mean Prior & Posterior RMSE

> python compute_rmse_layers.py <input_model.nc> <input_truth.nc> -v x -o <output_rmse_file.txt> -l \<line_label_in_output_text_file>

In [ ]:
cd $JEDI_EDU
output_filename="${EXPDIR}/plots/rmse_stream.txt"

# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
start_time=`date +%s` # Needed to calculate time it takes to complete rmse computation
for icyc in $(seq 1 $ncycles); do
   echo "Computing rmse for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_rmse_layers.py \
          ${EXPDIR}/da/letkf_meanprior.an.${curdate}.nc \
          ./CAPSTONE/data/truth/truth_reduced.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   # Run for analysis mean file
   python ./plots_scripts/compute_rmse_layers.py \
          ${EXPDIR}/da/letkf.000000.an.${curdate}.nc \
          ./CAPSTONE/data/truth/truth_reduced.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          --v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

end_time_total=`date +%s`
echo "SUCCESS completion $ncycles cycles' RMSE computation for $EXPNAME, took $(( end_time_total - start_time )) seconds"


### Compute Prior & Posterior Ensemble Variance
> Using `qg_ens_variance.x` program on prior and posterior ensemble member files


In [ ]:
cd $JEDI_EDU/
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
for anorbg in bg an; do
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")
   if [ "$anorbg" == "bg" ]; then
      zpad=0
      if [ $icyc -eq 1 ]; then
         bgfile_mem1="CAPSTONE/data/bg/bkgd_reduced.ens.1.${curdate_minus1day}.P1D.nc"
         bgfile="CAPSTONE/data/bg/bkgd_reduced.ens.%mem%.${curdate_minus1day}.P1D.nc"
      else
         bgfile_mem1="${EXPDIR}/da/letkf.1.fc.${curdate_minus1day}.P1D.nc"
         bgfile="${EXPDIR}/da/letkf.%mem%.fc.${curdate_minus1day}.P1D.nc"
      fi
   else
       zpad=6
       bgfile_mem1="${EXPDIR}/da/letkf.000000.an.${curdate}.nc"
       bgfile="${EXPDIR}/da/letkf.%mem%.an.${curdate}.nc"
   fi
   

   # Build namelist for ens variance JEDI program
   cat << EOF > ./CAPSTONE/yamls/ens_variance_cyc${icyc}.yaml
background:
  date: $curdate
  filename: $bgfile_mem1
  state variables: [x,q,u,v]
ensemble:
  members from template:
    template:
      date: $curdate
      filename: $bgfile
      state variables: [x,q,u,v]
    pattern: %mem%
    zero padding: $zpad
    nmembers: ${ENSSIZE}
geometry:
  nx: 20
  ny: 10
  depths: [4000.0, 6000.0]
variance output:
  datadir:  ${EXPDIR}/da
  exp: ${anorbg}EnsVariance
  type: diag
  date: $curdate
EOF
   # Run the ens variance program
   $JEDI_BIN/qg_ens_variance.x ./CAPSTONE/yamls/ens_variance_cyc${icyc}.yaml \
                            >& ${EXPDIR}/ens_var_cyc${icyc}.log

   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running qg_ens_variance.x for cycle $icyc $anorbg. Check ${EXPDIR}/ens_var_cyc${icyc}.log"
      echo "Stopping loops"
      break 2
   else
      echo "SUCCESS qg_ens_variance.x run for cycle $icyc $anorbg"
   fi
done
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

ls ${EXPDIR}/da/*EnsVariance*nc

### Calculate Average ensemble variance (prior and posterior) for each cycle

>Compute the 2D average variance for each layer, and for first guess and analysis ensemble, using `compute_ensvar_layers.py`. Save the values to the ensvar_stream.txt for this experiment.

In [ ]:
cd $JEDI_EDU
output_filename="${EXPDIR}/plots/ensvar_stream.txt"
# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing average ensemble variance for $curdate (cyc $icyc of $ncycles) . . . "
   # Run for background mean file
   python ./plots_scripts/compute_ensvar_layers.py \
          ${EXPDIR}/da/bgEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   python ./plots_scripts/compute_ensvar_layers.py \
          ${EXPDIR}/da/anEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

### Plot Sawtooth 

> The `plot_rmse_sawtooth_withvar.py` script has been provided to plot the sawtooth with ensemble spread overlaid

In [ ]:
cd $JEDI_EDU/plots_scripts

python plot_rmse_sawtooth_withvar2.py -m 8e7  $JEDI_EDU/${EXPDIR}/plots/rmse_stream.txt \
                             $JEDI_EDU/${EXPDIR}/plots/ensvar_stream.txt  \
                             -o $JEDI_EDU/${EXPDIR}/plots/sawtooth_${EXPNAME}.png